# Step 1 - Install the dependencies
- You can download and install the dependencies and all the libraries by executing the command below.
- Once the command is executed successfully. You need to restart the kernel to reflect all the changes. You can restart the kernel by clicking on the “Kernel" tab and then on the “Restart Kernel” option.

In [ ]:
!pip install -r requirements.txt

# Step 2 - Import the dataset
Execute this step if the dataset has been uploaded to an S3 bucket and you intend to retrieve it from the bucket.
- The code below downloads the dataset from the S3 bucket and stores it as `dataset.csv` in the notebook’s environment.

> While using this code, please replace the `<Bucket_Name>` placeholder with the actual name of your bucket.

In [ ]:
import boto3
from sagemaker import get_execution_role
role = get_execution_role()
print("The role attached is:",role)


s3_bucket = "<Bucket_Name>"
s3_key = "cleaned_courses_with_names.csv"

# Download dataset from S3
local_csv_path = "dataset.csv"
s3 = boto3.client("s3")
s3.download_file(s3_bucket, s3_key, local_csv_path)

# Load dataset
df = pd.read_csv(local_csv_path)
print(df.head())  # Check data format

# Step 3 - Train the model
- The code snippet below converts the data to a more relevant format for our use case.
- The function `generate_typeahead_pairs` creates pairs for each course entry. It maps initials of each course from a minimum of 3 alphabets up to its total length and expands it to the specific output. Let's say we have Python in our dataset, the expanded dataset would look like:

`[`
  <div>`{"input": "Pyt", "output": "Python"},` </div>
  <div>`{"input": "Pyth", "output": "Python"}, `</div>
  <div>`{"input": "Pytho", "output": "Python"}`</div>
`]`


- After generating the training pairs, the code transforms the dataset into a Hugging Face `Dataset` object, preparing it for model training.

- The `t5-small` model and tokenizer are loaded using the Hugging Face `transformers` library. Tokenization is handled via a simple preprocessing function that aligns the input with the expected output (target) for sequence-to-sequence training.

- The script uses `Trainer` and `TrainingArguments` from `transformers` to fine-tune the `t5-small` model. It supports GPU acceleration if available, and uses strategies like per-epoch evaluation, saving, and logging to ensure training quality.

- After training, the fine-tuned model and tokenizer are saved locally for future use.

***
We're using a seq2seq model because it's ideal for text generation tasks—like predicting full course names from partial input (e.g., `"Pyth"` → `"Python"`).

We chose `t5-small` as it's a lightweight, pre-trained model from the `T5` family, well-suited for quick fine-tuning on smaller datasets. Since `T5` treats all NLP tasks as `text-to-text`, it fits perfectly with our typeahead use case.

In [ ]:
import torch
from transformers import TrainingArguments, Trainer, AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import Dataset
import pandas as pd

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
df = pd.read_csv("dataset.csv")

def generate_typeahead_pairs(course_names, min_chars=3):
    typeahead_data = []
    for name in course_names:
        for i in range(min_chars, len(name)):
            input_text = name[:i]
            output_text = name
            typeahead_data.append({"input": input_text, "output": output_text})
    return typeahead_data

# Preprocess dataset
course_names = df["name"].tolist()
typeahead_pairs = generate_typeahead_pairs(course_names)
dataset = Dataset.from_pandas(pd.DataFrame(typeahead_pairs))

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

def tokenize_function(examples):
    return tokenizer(examples["input"], text_target=examples["output"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # Due to time constraint this is set to 1. The actual model was trained on 10 epochs.
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset
)

trainer.train()

# Save fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")
print("Training complete. Model saved successfully.")


Here's a brief explanation of the code:
- <b>Importing required libraries:</b> The script begins by importing essential libraries like `torch`, `transformers`, `datasets`, and `pandas`. These handle model loading, training, tokenization, and data preprocessing.

- <b>Checking GPU availability:</b> Then we check if a GPU is available to speed up training. If not, it defaults to using the CPU.

- <b>Loading the Dataset:</b> After that the dataset containing course names is loaded from a CSV file. This will be used to create the typeahead training examples.

- <b>Generating typeahead pairs:</b> `generate_typeahead_pairs` function is defined to create training pairs. For each course name, it generates multiple input-output pairs where the input is a prefix (like "Pyt") and the output is the full course name ("Python"). This simulates how a user might type part of a name and expect a suggestion.

- <b>Preparing the dataset:</b> The generated pairs are converted into a format compatible with HuggingFace’s Dataset class, which makes them easier to tokenize and train on.

- <b>Loading the model and tokenizer:</b> Then the pre-trained `t5-small` model and `tokenizer` are loaded.

- <b>Tokenizing the dataset:</b> We've defined a tokenization function `tokenize_function` to process each example, converting text into token IDs with padding and truncation to ensure consistent input length.

- <b>Defining training parameters:</b> After that we've defined the `Training` settings with learning rate, number of epochs, batch size, logging, and saving strategies. We've also enabled mixed-precision training (FP16) if a GPU is available, which can speed up training and reduce memory usage.

- <b>Training the model:</b> The Trainer class from HuggingFace is used to handle the full training loop, including `evaluation` and `checkpoint` saving.

- <b>Saving the Fine-Tuned Model:</b> Once training is complete, the fine-tuned model and tokenizer are saved to a local directory so they can be reused or deployed later.

# Step 4 - Model evaluation
- In the code snippet below loads the fine-tuned sequence-to-sequence model from the directory it is stored and sets it to evaluation mode. It then checks for GPU availability and moves the model to the appropriate device (GPU or CPU) for inference.

In [ ]:
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, AutoModelForSeq2SeqLM


model_path2 = "./fine_tuned_model2"  # Change this if stored elsewhere
model = AutoModelForSeq2SeqLM.from_pretrained(model_path2)
model.eval()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)


# Step 5 - Check model's perplexity

- In this code, we’re loading a fine-tuned T5 model and its tokenizer, moving it to the available device (GPU or CPU), and preparing a set of sample input texts. The model processes the tokenized inputs, computes the loss, and calculates perplexity—an important metric for evaluating language models—while printing the results.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load fine-tuned T5 model and tokenizer
model_path = "./fine_tuned_model2"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Sample input texts
test_texts = ["Machine Learning", "Deep Learning", "Neural Networks"]
tokenized_inputs = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True)

# Move tokenized inputs to the same device as model
tokenized_inputs = {key: val.to(device) for key, val in tokenized_inputs.items()}

# Compute loss and perplexity
with torch.no_grad():
    outputs = model(**tokenized_inputs, labels=tokenized_inputs["input_ids"])
    loss = outputs.loss
    perplexity = torch.exp(loss).item()

print(f"Model Perplexity: {perplexity}")
print(f"Outputs: {len(outputs)}")


# Step 6 - Test the model
- Here, we're using the fine-tuned model with a text-to-text generation pipeline to provide typeahead suggestions based on a partial input ("de"). The model generates five diverse suggestions with a maximum length of `50` tokens, using techniques like sampling and nucleus sampling to ensure variety in the outputs. Finally, the code prints each generated suggestion for review.

In [ ]:
from transformers import pipeline

# Load fine-tuned model with the correct pipeline for T5
generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Provide partial input for typeahead suggestion
input_text = "de"

# Generate multiple suggestions with sampling enabled
suggestions = generator(
    input_text,
    num_return_sequences=5,  # Generate 5 different outputs
    max_length=50,           # Limit response length
    do_sample=True,          # Enable sampling for diverse suggestions
    top_k=50,                # Consider top 50 tokens at each step
    top_p=0.95               # Use nucleus sampling for better diversity
)

# Print results
for i, suggestion in enumerate(suggestions):
    print(f"Suggestion {i+1}: {suggestion['generated_text']}")


# Step 7 - Generate tar zip file and upload it to s3
- The commands below are used to create a `typeahead_model.tar.gz` and uplaod the zip file to the S3 bucket.
***
While using this code please ensure that you replace the `<Bucket_Name>` placeholder with the actual name of your bucket.

In [ ]:
!tar -czvf typeahead_model.tar.gz -C fine_tuned_model2 .
!aws s3 cp typeahead_model.tar.gz s3://<Bucket_Name>/typeahead_model/

# Step 8 - Create a directory and store important files

***
Before executing this command please ensure that you have uploaded the `inference.py` and `requirements.txt` file to your notebook instance. These files are provided in the `Notebook files` playground in the lesson.
***
-  The commands below create a new directory in the notebook environment and copy the `inference.py` and `requirements.txt` to it.

In [ ]:
!mkdir code
!cp inference.py requirements.txt code/

# Step 9 - Create model endpoint

In the following code snippet, we are setting up and deploying a PyTorch model on Amazon SageMaker:

- First, we retrieve the execution role and initialize a SageMaker session, which will allow us to interact with SageMaker's services and resources.

- Next, we define the location of the pre-trained model stored in an S3 bucket as a `.tar.gz` file and specify how to load it using the `PyTorchModel` class.

- We then configure the model deployment by specifying the entry point script (`inference.py`), the source directory containing the necessary code, and any dependencies listed in `requirements.txt`.

- Finally, we deploy the model to a SageMaker endpoint, specifying the instance type (`ml.g4dn.xlarge`) and the endpoint name (`typeahead-suggester`) that will be used for real-time predictions.

This setup allows us to easily send requests to the deployed model via the SageMaker endpoint and get responses for tasks such as typeahead suggestions.










In [ ]:
import sagemaker
from sagemaker.pytorch.model import PyTorchModel
from sagemaker import get_execution_role

role = get_execution_role()
sagemaker_session = sagemaker.Session()
bucket = "typeaheadbucket-haris"
model_artifact = f"s3://{bucket}/typeahead_modelv2/typeahead_model.tar.gz"

pytorch_model = PyTorchModel(
    entry_point="inference.py",
    model_data=model_artifact,
    role=role,
    framework_version="2.2",
    py_version="py310",
    env={"PYTHONPATH": "."},
    source_dir="code",
    dependencies=['requirements.txt'],
)

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name="typeahead-suggester"
)

# Step 10 - Test the endpoint
In the following code snippet, we are interacting with a deployed model on Amazon SageMaker to make predictions:

- First, we import the `Predictor` class from the SageMaker SDK and prepare the `test_input` dictionary, which contains the text for which we want to generate a typeahead suggestion.

- We then create a `Predictor` object that connects to the SageMaker endpoint (`typeahead-suggester`) that was previously deployed.

- Next, we send the input text to the model by calling the `predict()` method of the `Predictor` object. We convert the `test_input` into a `JSON` format and specify that the content type is `"application/json"`.

- Finally, we print the model's response, decoding it if it's in bytes format. The response contains the model's prediction based on the input provided.

This code allows us to interact with the deployed model and retrieve real-time predictions for typeahead suggestions.

In [ ]:
from sagemaker.predictor import Predictor
import json

predictor = Predictor(endpoint_name="typeahead-suggester")

test_input = {"text": "infra"}

# Call the endpoint
response = predictor.predict(
    json.dumps(test_input),
    initial_args={"ContentType": "application/json"}
)

print(response.decode("utf-8") if isinstance(response, bytes) else response)
